# Run pipeline stages (thin notebook; PLAN §3)

**Notebook settings:** Accelerator **GPU T4 x2** · Internet **on** · Secrets: **HF_TOKEN**,
**GITHUB_TOKEN** (only if the repo is private). Set the parameters in the next cell, then run all.
Every stage skips itself on a valid manifest and resumes an incomplete one, so re-running after a
12 h cutoff continues where it stopped.

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "m0-green"                    # a branch, a tag, or a FULL 40-character commit SHA
CONFIG = "configs/smoke.yaml"
STAGES = "sentences,embed"          # comma-separated; "" runs all stages
EXTRA = "train"                     # "train" (embed/finetune) or "bench" (generate/score)
WORK = "/kaggle/working/rbbd"

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
!pip install -q -e ".[{EXTRA},dev]" 2>&1 | tail -15
!pip freeze > /kaggle/working/pip_freeze_{REF}.txt

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
_stages = f"--stages {STAGES}" if STAGES else ""
!python -m rbbd.cli run --config {CONFIG} {_stages}
!python -m rbbd.cli status --config {CONFIG}

In [ ]:
# Mirror what this session produced into the private store (D-041).
for _path in ["manifests", *[s for s in STAGES.split(",") if s]]:
    if os.path.isdir(f"/kaggle/working/artifacts/{_path}"):
        !python -m rbbd.cli sync --config {CONFIG} --path {_path}
!python -m rbbd.cli sync --config {CONFIG} --path env/$RBBD_SESSION_ID